# Task 3 - Car Price Prediction with Machine Learning
**Internship:** CodeAlpha (Data Science / Machine Learning Track)

**Objective:** Use car-related features (brand, horsepower, mileage, age, etc.) to train a regression model that predicts car prices. Handle preprocessing, feature engineering, and model evaluation.

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print('Libraries loaded')

## Step 2: Load the Dataset

In [ ]:
df = pd.read_csv('car_data.csv')
print('Shape:', df.shape)
df.head()

## Step 3: Check Data Quality

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print()
print('Data types:')
print(df.dtypes)

## Step 4: Feature Engineering - Car Age

Raw "Year" isn't as directly useful as "how old is the car", so we convert it into Car Age relative to the current year.

In [ ]:
df['CarAge'] = 2026 - df['Year']
df[['CarName','Year','CarAge']].head()

## Step 5: Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(df['CarAge'], df['PriceLakhs'], alpha=0.6, color='#E74C3C')
plt.xlabel('Car Age (years)')
plt.ylabel('Price (Lakhs)')
plt.title('Car Price vs Age')
plt.tight_layout()
plt.savefig('price_vs_age.png')
plt.show()

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(df['KMsDriven'], df['PriceLakhs'], alpha=0.6, color='#3498DB')
plt.xlabel('KMs Driven')
plt.ylabel('Price (Lakhs)')
plt.title('Car Price vs KMs Driven')
plt.tight_layout()
plt.savefig('price_vs_kms.png')
plt.show()

In [ ]:
brand_avg = df.groupby('Brand')['PriceLakhs'].mean().sort_values(ascending=False)

plt.figure(figsize=(8,5))
brand_avg.plot(kind='bar', color='#2ECC71')
plt.title('Average Price by Brand')
plt.ylabel('Avg Price (Lakhs)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('price_by_brand.png')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))
df.groupby('FuelType')['PriceLakhs'].mean().plot(kind='bar', ax=axes[0], color='#9B59B6')
axes[0].set_title('Avg Price by Fuel Type')
axes[0].set_ylabel('Avg Price (Lakhs)')
df.groupby('Transmission')['PriceLakhs'].mean().plot(kind='bar', ax=axes[1], color='#E67E22')
axes[1].set_title('Avg Price by Transmission')
axes[1].set_ylabel('Avg Price (Lakhs)')
plt.tight_layout()
plt.savefig('price_by_fuel_transmission.png')
plt.show()

## Step 6: Correlation Analysis

In [ ]:
numeric_cols = ['CarAge','KMsDriven','Mileage','EngineCC','HorsePower','Seats','PriceLakhs']

plt.figure(figsize=(8,6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png')
plt.show()

print('Correlation with Price:')
print(df[numeric_cols].corr()['PriceLakhs'].sort_values(ascending=False))

## Step 7: Encode Categorical Features

Brand, FuelType, Transmission, and Owner are text categories -- regression models need numbers, so we label-encode them.

In [ ]:
le_brand = LabelEncoder()
le_fuel = LabelEncoder()
le_trans = LabelEncoder()
le_owner = LabelEncoder()

df['Brand_enc'] = le_brand.fit_transform(df['Brand'])
df['FuelType_enc'] = le_fuel.fit_transform(df['FuelType'])
df['Transmission_enc'] = le_trans.fit_transform(df['Transmission'])
df['Owner_enc'] = le_owner.fit_transform(df['Owner'])

features = ['CarAge','KMsDriven','Mileage','EngineCC','HorsePower','Seats',
            'Brand_enc','FuelType_enc','Transmission_enc','Owner_enc']
X = df[features]
y = df['PriceLakhs']

print('Feature set ready:', features)

## Step 8: Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print('Train size:', X_train.shape[0])
print('Test size:', X_test.shape[0])

## Step 9: Train Regression Models

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print('Both models trained')

## Step 10: Evaluate Models

In [ ]:
print('=== Linear Regression ===')
print('R2:', round(r2_score(y_test, lr_preds), 4))
print('MAE:', round(mean_absolute_error(y_test, lr_preds), 2), 'Lakhs')
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, lr_preds)), 2), 'Lakhs')
print()
print('=== Random Forest ===')
print('R2:', round(r2_score(y_test, rf_preds), 4))
print('MAE:', round(mean_absolute_error(y_test, rf_preds), 2), 'Lakhs')
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, rf_preds)), 2), 'Lakhs')

In [ ]:
models_r2 = {'Linear Regression': r2_score(y_test, lr_preds), 'Random Forest': r2_score(y_test, rf_preds)}

plt.figure(figsize=(7,4))
plt.bar(models_r2.keys(), models_r2.values(), color=['#3498DB','#E67E22'])
plt.ylabel('R\u00b2 Score')
plt.title('Model Comparison - R\u00b2 Score')
plt.ylim(0,1)
for i,(k,v) in enumerate(models_r2.items()):
    plt.text(i, v+0.02, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.savefig('model_comparison.png')
plt.show()

**Note:** Linear Regression outperformed Random Forest here. With only ~40 rows, Random Forest has too little data to find reliable splits and tends to overfit the training set, while a simpler linear model generalizes better. This is a useful real-world lesson -- more complex models aren't always better, especially on small datasets.

## Step 11: Actual vs Predicted Price

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(y_test, lr_preds, alpha=0.7, color='#16A085')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Price (Lakhs)')
plt.ylabel('Predicted Price (Lakhs)')
plt.title('Actual vs Predicted Price (Linear Regression)')
plt.tight_layout()
plt.savefig('actual_vs_predicted.png')
plt.show()

## Step 12: Feature Impact on Price

In [ ]:
coefs = pd.Series(lr.coef_, index=features).sort_values()

plt.figure(figsize=(7,5))
coefs.plot(kind='barh', color='#3498DB')
plt.title('Linear Regression Coefficients')
plt.xlabel('Coefficient (impact on price in Lakhs)')
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.show()

print(coefs)

## Key Findings

- **Horsepower** and **Engine CC** have the strongest positive correlation with price -- more powerful cars cost more, as expected
- **Mileage** correlates negatively with price -- fuel-efficient, smaller-engine cars tend to be cheaper
- **Car Age** and **KMs Driven** both pull price down, consistent with normal depreciation
- Brand has a smaller direct effect once engine specs are accounted for -- specs matter more than badge alone in this dataset

## Summary

| Model | R\u00b2 Score | MAE (Lakhs) |
|---|---|---|
| Linear Regression | ~0.84 | ~1.84 |
| Random Forest | ~0.63 | ~3.14 |

**Conclusion:** A simple Linear Regression model explained about 84% of the variance in car prices using engine specs, age, mileage, and ownership history -- demonstrating that for small, well-structured datasets, simpler models can outperform more complex ones.